# DICOM Quickcheck - XNAT

Batch QC review of DICOM data from an XNAT project.

**Features:**
- Save/load progress to avoid re-processing
- Interactive progress with ETA
- Incremental updates (only process new scans)

**Setup:** Upload `dicom_qc.zip` to your workspace before running.

In [ ]:
import sys
import os
import shutil
import zipfile
from pathlib import Path

WORKSPACE = Path.cwd()

# Extract dicom_qc (overwrite if exists)
zip_path = WORKSPACE / 'dicom_qc.zip'
pkg_path = WORKSPACE / 'dicom_qc'
if zip_path.exists():
    if pkg_path.exists():
        shutil.rmtree(pkg_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(WORKSPACE)
    print('Extracted dicom_qc.zip')

sys.path.insert(0, str(WORKSPACE))

%matplotlib widget

In [ ]:
# Connect to XNAT and select project
import xnat

session = xnat.connect()
project = session.projects[os.environ['XNAT_PROJECT']]
print(f"Project: {project.name} ({len(project.subjects)} subjects)")

In [ ]:
from dicom_qc import QuickCheck

# Save file for this project
SAVE_FILE = WORKSPACE / f'qc_{os.environ["XNAT_PROJECT"]}_state.pkl'

# Load existing progress or create new
if SAVE_FILE.exists():
    print(f"Loading saved state from {SAVE_FILE}")
    qc = QuickCheck.from_save(SAVE_FILE)
else:
    print("Starting fresh (no saved state found)")
    qc = QuickCheck()

# Connect XNAT session (required for file access)
qc.connect_xnat(session)
qc._save_path = SAVE_FILE

In [ ]:
# Discover scans from XNAT (skips existing, adds new)
qc.discover_xnat(project, refresh=False)

In [ ]:
# Process series with live progress (auto-saves every 10 series)
results = qc.process_all_interactive()

In [ ]:
# Generate HTML report and save final state
report_path = WORKSPACE / f'qc_{os.environ["XNAT_PROJECT"]}_report.html'
qc.generate_html_report(report_path)
qc.save()
print(f"Report: {report_path}")

In [ ]:
# Interactive review
qc.display()